# Khoảng cách tiếp cận điện & internet, 2000–2020
### CSCU9M3 · Autumn 2022 · Số sinh viên: **ZZZZZZZ**

> ⚠️ **Đây là bộ khung tham khảo (skeleton)** dựng theo `HUONG_DAN_TUNG_CELL_HUONG_A.md`.
> - Mọi chỗ đánh dấu **`[TODO]`** do **bạn** viết — đây là phần được chấm.
> - Các cell **plumbing** (đọc file, làm sạch, kiểm tra) đã có sẵn để tiết kiệm thời gian —
>   bạn phải **hiểu từng dòng** và giải thích lại bằng lời của mình.
> - Cell nào chỉ có comment `[TODO]` sẽ chưa có output → bạn điền code rồi chạy.
> - Trước khi nộp: thay toàn bộ `[TODO]` bằng nội dung thật, chạy `Restart & Run All`,
>   xoá mọi ghi chú khung này, đổi tên file `ZZZZZZZ_M3_Aut22_TheAssignment.ipynb`.

## Executive Summary

**[TODO — viết SAU CÙNG · 1 cell markdown DUY NHẤT · ≤ 200 từ]**

Cấu trúc gợi ý cho 200 từ:
1. Bộ dữ liệu + phạm vi (217 quốc gia, 2000–2020, 5 chỉ số).
2. 2–3 phát hiện chính — mỗi phát hiện kèm **con số cụ thể**.
3. Một câu hàm ý chính sách.

## Introduction

**[TODO — ≤ 200 từ, 3 đoạn]**

1. **Bộ dữ liệu là gì**: SDG Indicators của World Bank, 1990–2020, tải từ đâu, gồm 6 file CSV.
2. **Vì sao chọn đề tài này**: khoảng cách điện & internet = SDG 7 & 9, nơi bất bình đẳng thể hiện rõ nhất.
3. **Kỳ vọng phát hiện gì** (khác với kết quả thật cũng được — template chấp nhận).

*Không liệt kê chi tiết từng file ở đây — để sang mục Data Cleaning.*

In [1]:
# ── CELL 3: chốt phạm vi 5 chỉ số, dựa trên metadata (không chọn bừa) ──
import pandas as pd

INDICATORS = {
    'EG.ELC.ACCS.ZS':    'Access to electricity (% of population)',
    'EG.CFT.ACCS.ZS':    'Access to clean fuels for cooking (% of population)',
    'IT.NET.USER.ZS':    'Individuals using the Internet (% of population)',
    'SP.URB.TOTL.IN.ZS': 'Urban population (% of total population)',
    'NY.GDP.PCAP.CD':    'GDP per capita (current US$)',
}
selected_codes = list(INDICATORS)

df_series = pd.read_csv('Dataset/SDGSeries.csv')
meta = df_series[df_series['Series Code'].isin(selected_codes)][
    ['Series Code', 'Topic', 'Indicator Name', 'Source', 'Periodicity', 'Aggregation method']
]
print(f"Metadata tìm thấy cho {len(meta)}/{len(selected_codes)} mã chỉ số:")
display(meta.reset_index(drop=True))

Metadata tìm thấy cho 5/5 mã chỉ số:


,Series Code,Topic,Indicator Name,Source,Periodicity,Aggregation method
0,EG.CFT.ACCS.ZS,Environment: Energy production & use,Access to clean fuels and technologies for coo...,WHO Global Health Observatory (https://www.wh...,Annual,Weighted average
1,EG.ELC.ACCS.ZS,Environment: Energy production & use,Access to electricity (% of population),World Bank Global Electrification Database fro...,Annual,Weighted average
2,IT.NET.USER.ZS,Infrastructure: Communications,Individuals using the Internet (% of population),International Telecommunication Union (ITU) Wo...,Annual,Weighted average
3,NY.GDP.PCAP.CD,Economic Policy & Debt: National accounts: US$...,GDP per capita (current US$),"World Bank national accounts data, and OECD Na...",Annual,Weighted average
4,SP.URB.TOTL.IN.ZS,Environment: Density & urbanization,Urban population (% of total population),United Nations Population Division. World Urba...,Annual,Weighted average


**[TODO — diễn giải bảng metadata]**

- Mỗi chỉ số thuộc **Topic** gì, **nguồn** nào (ITU / WHO GHO / UN Population Division / World Bank)?
- Vì sao 5 chỉ số này ghép được thành **một** câu chuyện?
- **Ghi chú nguồn ngay đây** (rule C4) — sẽ liệt kê lại ở Bibliography.

*Bẫy:* cột `Unit of measure` trong `SDGSeries.csv` để trống → đừng kết luận "không có đơn vị";
đơn vị đọc từ `Indicator Name`: 4 cái đầu là **%**, cái cuối là **US$**.

## Data Cleaning

**[TODO — kế hoạch làm sạch: 6–7 bullet, mỗi bước kèm LÝ DO]**

Gợi ý thứ tự: đọc file → xử lý cột thừa → tách aggregate → melt wide→long →
lọc chỉ số → phân tích missing → chốt cửa sổ thời gian → join metadata.

*(Viết kế hoạch TRƯỚC khi code → examiner thấy bạn có kế hoạch, không chọc ngoáy.)*

In [2]:
# ── CELL 6: đọc SDGData.csv & khảo sát ban đầu ──
df_raw = pd.read_csv('Dataset/SDGData.csv')

print("Shape:", df_raw.shape)                  # kỳ vọng (106488, 36)
print("Cột:", df_raw.columns.tolist())         # cột cuối tên rỗng → 'Unnamed: 35'
print("\nKiểu dữ liệu:")
print(df_raw.dtypes)
display(df_raw.head(3))

Shape: (106488, 36)
Cột: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', 'Unnamed: 35']

Kiểu dữ liệu:
Country Name       object
Country Code       object
Indicator Name     object
Indicator Code     object
1990              float64
1991              float64
1992              float64
1993              float64
1994              float64
1995              float64
1996              float64
1997              float64
1998              float64
1999              float64
2000              float64
2001              float64
2002              float64
2003              float64
2004              float64
2005              float64
2006              float64
2007              float64
2008              float64
2009              float64
2010

,Country Name,Country Code,Indicator Name,Indicator Code,1990,1991,1992,1993,1994,1995,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,Unnamed: 35
0,Arab World,ARB,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,85.644218,85.932567,86.232384,86.478597,86.722685,86.937933,87.040774,87.235539,87.307068,NaN
1,Arab World,ARB,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,87.039588,88.992620,88.015356,88.681886,89.195062,90.324659,88.910749,89.999946,90.277735,NaN
2,Arab World,ARB,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,NaN,NaN,54.722944,56.807039,57.872605,57.447725,...,75.377022,79.582622,77.666157,78.951592,79.791665,82.373066,82.970754,83.740500,81.660899,NaN


**[TODO — 3 phát hiện từ cell trên]**

- (a) Dữ liệu đang ở dạng **wide** — mỗi năm một cột.
- (b) Có **1 cột thừa** ở cuối bảng (dấu phẩy thừa).
- (c) Các cột năm đang là **chuỗi (object)** → phải ép sang số trước khi tính.

In [3]:
# ── CELL 8: xoá cột thừa + ép 31 cột năm sang numeric ──
df_data = df_raw.drop(
    columns=[c for c in df_raw.columns if c.startswith('Unnamed')], errors='ignore'
)
year_cols = [c for c in df_data.columns if c.isdigit()]        # '1990' … '2020'

# Chỉ ép 31 CỘT NĂM — ép nhầm 4 cột chữ đầu sẽ thành NaN hàng loạt
for col in year_cols:
    df_data[col] = pd.to_numeric(df_data[col], errors='coerce')

print("Shape sau xử lý:", df_data.shape)        # kỳ vọng (106488, 35)
print("Số cột năm:", len(year_cols))           # kỳ vọng 31
print("\nKiểu dữ liệu vài cột năm sau khi ép:")
print(df_data[year_cols[:5]].dtypes)

Shape sau xử lý: (106488, 35)
Số cột năm: 31

Kiểu dữ liệu vài cột năm sau khi ép:
1990    float64
1991    float64
1992    float64
1993    float64
1994    float64
dtype: object


In [4]:
# ── CELL 9: đọc SDGCountry.csv & dựng cờ quốc gia thật / vùng tổng hợp ──
df_country = pd.read_csv('Dataset/SDGCountry.csv')

# Region KHÔNG rỗng → quốc gia thật; Region rỗng → vùng/tổng hợp
df_country['is_country'] = df_country['Region'].notna()

counts = df_country['is_country'].value_counts()
print("Quốc gia thật:", counts.get(True, 0))     # kỳ vọng 217
print("Vùng tổng hợp:", counts.get(False, 0))    # kỳ vọng 44

aggregates = df_country.loc[~df_country['is_country'], 'Short Name'].tolist()
print("\nDanh sách 44 vùng/tổng hợp:")
for i, name in enumerate(aggregates, 1):
    print(f"{i:2d}. {name}")

Quốc gia thật: 217
Vùng tổng hợp: 44

Danh sách 44 vùng/tổng hợp:
 1. Arab World
 2. Central Europe and the Baltics
 3. Caribbean small states
 4. East Asia & Pacific (excluding high income)
 5. Early-demographic dividend
 6. East Asia & Pacific
 7. Europe & Central Asia (excluding high income)
 8. Europe & Central Asia
 9. Euro area
10. European Union
11. High income
12. Heavily indebted poor countries (HIPC)
13. IBRD only
14. IDA & IBRD total
15. IDA total
16. IDA blend
17. IDA only
18. Latin America & Caribbean (excluding high income)
19. Latin America & Caribbean
20. Least developed countries: UN classification
21. Low income
22. Lower middle income
23. Low & middle income
24. Late-demographic dividend
25. Middle East & North Africa
26. Middle income
27. Middle East & North Africa (excluding high income)
28. North America
29. OECD members
30. Other small states
31. Pre-demographic dividend
32. Post-demographic dividend
33. South Asia
34. Sub-Saharan Africa (excluding high income)
3

**[TODO — vì sao PHẢI lọc aggregate (lỗi nghiêm trọng nhất của bài)]**

Nếu giữ `World` + `European Union` + `High income` + các nước rồi tính trung bình →
**đếm trùng dữ liệu**, mọi con số trung bình đều sai.

**Bẫy tên:** tồn tại thực thể tổng hợp tên **`East Asia & Pacific`** trùng hệt tên một
`Region` của quốc gia thật → **lọc theo `Country Code`**, KHÔNG lọc theo tên.

In [5]:
# ── CELL 11: lọc 217 nước thật + 5 chỉ số, rồi xoay wide → long ──
valid_codes = df_country.loc[df_country['is_country'], 'Country Code']

df_sub = df_data[
    df_data['Country Code'].isin(valid_codes)
    & df_data['Indicator Code'].isin(selected_codes)
].copy()
print("Shape wide sau lọc:", df_sub.shape)       # kỳ vọng (1085, 35) = 217 × 5

# Wide (31 cột năm) → Long (mỗi dòng = 1 (nước, chỉ số, năm))
df_long = df_sub.melt(
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    var_name='Year', value_name='Value'
)
df_long['Year'] = df_long['Year'].astype(int)

print("Shape long sau melt:", df_long.shape)     # kỳ vọng (33635, 6) = 1085 × 31
display(df_long.head())

Shape wide sau lọc: (1085, 35)
Shape long sau melt: (33635, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
0,Afghanistan,AFG,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,1990,NaN
1,Afghanistan,AFG,Access to electricity (% of population),EG.ELC.ACCS.ZS,1990,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,1990,NaN
3,Afghanistan,AFG,Individuals using the Internet (% of population),IT.NET.USER.ZS,1990,0.000
4,Afghanistan,AFG,Urban population (% of total population),SP.URB.TOTL.IN.ZS,1990,21.177


In [6]:
# ── CELL 12: khảo sát missing theo NĂM và theo CHỈ SỐ (subset 5 chỉ số) ──
coverage_year = df_long.groupby('Year')['Value'].agg(non_null='count', total='size')
coverage_year['pct_thieu'] = (100 * (1 - coverage_year['non_null'] / coverage_year['total'])).round(1)
display(coverage_year)

coverage_ind = df_long.groupby('Indicator Code')['Value'].agg(non_null='count', total='size')
coverage_ind['pct_thieu'] = (100 * (1 - coverage_ind['non_null'] / coverage_ind['total'])).round(1)
display(coverage_ind.sort_values('pct_thieu', ascending=False))

# [TODO] Vẽ hình cho Quan sát 1: số giá trị CÓ DỮ LIỆU theo năm
# Gợi ý: 2 đường — toàn bộ SDGData vs subset 5 chỉ số (độ phủ tăng dần rồi rơi ở 2020)
# Nguồn chú thích: số liệu 2020 chịu ảnh hưởng COVID (trích SDGSeries-Time.csv)

,non_null,total,pct_thieu
Year,,,
1990,699,1085,35.6
1991,710,1085,34.6
1992,724,1085,33.3
1993,735,1085,32.3
1994,743,1085,31.5
1995,760,1085,30.0
1996,732,1085,32.5
1997,748,1085,31.1
1998,762,1085,29.8


,non_null,total,pct_thieu
Indicator Code,,,
EG.CFT.ACCS.ZS,3986,6727,40.7
EG.ELC.ACCS.ZS,5896,6727,12.4
IT.NET.USER.ZS,6068,6727,9.8
NY.GDP.PCAP.CD,6202,6727,7.8
SP.URB.TOTL.IN.ZS,6665,6727,0.9


In [7]:
# ── CELL 13: kiểm tra chất lượng dữ liệu tổng quát ──

# (a) Duplicate theo cặp (Country Code, Indicator Code) trên TOÀN BỘ dữ liệu
n_dup = df_data.duplicated(subset=['Country Code', 'Indicator Code']).sum()
print(f"(a) Số dòng duplicate: {n_dup}")          # kỳ vọng 0

# (b) Chỉ số rỗng 100% (mọi nước × mọi năm)
null_by_ind = df_data.groupby('Indicator Code')[year_cols].apply(lambda g: g.isna().all().all())
fully_empty = null_by_ind[null_by_ind].index.tolist()
print(f"(b) {len(fully_empty)} chỉ số rỗng 100%: {fully_empty}")
#  Kỳ vọng 4: NE.CON.GOVT.KD, SL.ISV.IFRM.ZS, SI.POV.RUHC, SI.POV.URHC

# (c) Mã có trong data nhưng KHÔNG có metadata trong SDGSeries.csv
no_meta = sorted(set(df_data['Indicator Code']) - set(df_series['Series Code']))
print(f"(c) Mã thiếu metadata: {no_meta}")        # kỳ vọng ['NE.CON.GOVT.KD']

# (d) Chỉ số đơn vị % phải nằm trong [0, 100] (loại NaN khỏi phép check)
pct_codes = [c for c in selected_codes if c != 'NY.GDP.PCAP.CD']
val = df_long['Value']
out_of_range = df_long[df_long['Indicator Code'].isin(pct_codes)
                       & val.notna() & ~val.between(0, 100)]
print(f"(d) Giá trị % ngoài [0,100]: {len(out_of_range)} dòng")
display(out_of_range.head())

(a) Số dòng duplicate: 0
(b) 4 chỉ số rỗng 100%: ['NE.CON.GOVT.KD', 'SI.POV.RUHC', 'SI.POV.URHC', 'SL.ISV.IFRM.ZS']
(c) Mã thiếu metadata: ['NE.CON.GOVT.KD']


(d) Giá trị % ngoài [0,100]: 0 dòng


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value


**[TODO — quyết định về missing (phần được chấm nhiều) — ghi rõ 4 điểm]**

1. **Không impute** (không điền trung bình) — vì sao? (Missing không ngẫu nhiên, thiếu nhiều ở nước nghèo.)
2. Chốt cửa sổ thời gian **2000–2020** — vì sao? (`EG.CFT.ACCS.ZS` bắt đầu đúng 2000; thập niên 1990 mỏng.)
3. Internet dùng mốc **2015** làm năm cuối so sánh — vì sao? (Độ phủ 2020 chỉ 151/217.)
   Ghi rõ đây là **hạn chế dữ liệu**, không phải lỗi phân tích.
4. **Venezuela (VEN)** không có `Income Group` — bạn chọn cách nào? Nhất quán từ đầu.

In [8]:
# ── CELL 15: dựng bảng phân tích cuối (2 mốc năm + metadata quốc gia) ──
YEAR_START, YEAR_END, YEAR_NET = 2000, 2020, 2015   # YEAR_NET: mốc cuối cho internet

df_focus = df_long[df_long['Year'].isin([YEAR_START, YEAR_END, YEAR_NET])]

# Wide: mỗi (nước, chỉ số) 1 dòng, cột = chỉ số × năm
tbl = df_focus.pivot_table(
    index=['Country Name', 'Country Code'],
    columns=['Indicator Code', 'Year'],
    values='Value'
)
tbl.columns = [f'{ind}_{yr}' for ind, yr in tbl.columns]
tbl = tbl.reset_index()

# Join Region / Income Group từ metadata quốc gia
tbl = tbl.merge(
    df_country[['Country Code', 'Region', 'Income Group']],
    on='Country Code', how='left'
)

# TODO (Cell 14): uncomment ĐÚNG 1 dòng theo quyết định của bạn:
# tbl['Income Group'] = tbl['Income Group'].fillna('No income group reported')   # Cách A
# tbl = tbl[tbl['Income Group'].notna()].copy()                                  # Cách B

print("Shape bảng phân tích:", tbl.shape)
missing = tbl.isna().sum()
print("\nGiá trị thiếu còn lại theo cột:")
print(missing[missing > 0])
display(tbl.head())
display(tbl.describe())

Shape bảng phân tích: (217, 19)

Giá trị thiếu còn lại theo cột:
EG.CFT.ACCS.ZS_2000       27
EG.CFT.ACCS.ZS_2015       27
EG.CFT.ACCS.ZS_2020       28
EG.ELC.ACCS.ZS_2000        6
EG.ELC.ACCS.ZS_2015        1
EG.ELC.ACCS.ZS_2020        2
IT.NET.USER.ZS_2000       21
IT.NET.USER.ZS_2015       17
IT.NET.USER.ZS_2020       66
NY.GDP.PCAP.CD_2000       18
NY.GDP.PCAP.CD_2015        7
NY.GDP.PCAP.CD_2020       14
SP.URB.TOTL.IN.ZS_2000     2
SP.URB.TOTL.IN.ZS_2015     2
SP.URB.TOTL.IN.ZS_2020     2
Income Group               1
dtype: int64


,Country Name,Country Code,EG.CFT.ACCS.ZS_2000,EG.CFT.ACCS.ZS_2015,EG.CFT.ACCS.ZS_2020,EG.ELC.ACCS.ZS_2000,EG.ELC.ACCS.ZS_2015,EG.ELC.ACCS.ZS_2020,IT.NET.USER.ZS_2000,IT.NET.USER.ZS_2015,IT.NET.USER.ZS_2020,NY.GDP.PCAP.CD_2000,NY.GDP.PCAP.CD_2015,NY.GDP.PCAP.CD_2020,SP.URB.TOTL.IN.ZS_2000,SP.URB.TOTL.IN.ZS_2015,SP.URB.TOTL.IN.ZS_2020,Region,Income Group
0,Afghanistan,AFG,6.2,27.4,33.2,1.613591,71.500000,97.699997,NaN,8.26,18.40,NaN,556.007221,516.747871,22.078,24.803,26.026,South Asia,Low income
1,Albania,ALB,38.2,76.2,81.3,100.000000,99.980003,100.000000,0.114097,56.90,72.24,1126.683340,3952.802538,5332.160475,41.741,57.434,62.112,Europe & Central Asia,Upper middle income
2,Algeria,DZA,97.1,99.5,99.7,98.973099,99.186661,99.804131,0.491706,38.20,62.90,1765.027146,4177.889542,3306.858208,59.919,70.848,73.733,Middle East & North Africa,Lower middle income
3,American Samoa,ASM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12059.635165,12844.900991,88.587,87.238,87.153,East Asia & Pacific,Upper middle income
4,Andorra,AND,100.0,100.0,100.0,100.000000,100.000000,100.000000,10.538836,87.90,NaN,21854.246803,35770.918653,37416.697720,92.395,88.345,87.916,Europe & Central Asia,High income


,EG.CFT.ACCS.ZS_2000,EG.CFT.ACCS.ZS_2015,EG.CFT.ACCS.ZS_2020,EG.ELC.ACCS.ZS_2000,EG.ELC.ACCS.ZS_2015,EG.ELC.ACCS.ZS_2020,IT.NET.USER.ZS_2000,IT.NET.USER.ZS_2015,IT.NET.USER.ZS_2020,NY.GDP.PCAP.CD_2000,NY.GDP.PCAP.CD_2015,NY.GDP.PCAP.CD_2020,SP.URB.TOTL.IN.ZS_2000,SP.URB.TOTL.IN.ZS_2015,SP.URB.TOTL.IN.ZS_2020
count,190.000000,190.000000,189.000000,211.000000,216.000000,215.000000,196.000000,200.000000,151.000000,199.000000,210.000000,203.000000,215.000000,215.000000,215.000000
mean,57.671368,65.581158,67.399735,75.672378,83.452441,86.953360,8.732304,48.034010,64.623466,8312.053771,16687.650764,16303.997091,55.491195,59.923451,61.415721
std,40.492401,38.468605,37.703206,33.498639,26.875029,23.605581,13.321321,28.997884,26.601769,13055.941231,24896.974671,23593.997636,24.612698,24.103121,23.858148
min,0.100000,0.000000,0.000000,1.613591,4.601340,7.241338,0.000000,1.083733,6.500000,124.460791,305.511144,233.837510,8.246000,12.078000,13.345000
25%,13.850000,26.137500,32.200000,54.012449,73.811436,86.136776,0.380223,20.916073,39.400000,651.272878,2213.957780,2262.250712,34.821000,40.914500,42.791500
50%,66.155000,85.450000,86.800000,97.021797,99.634331,100.000000,2.322545,49.927812,71.968194,2058.264401,6244.470178,6117.493571,55.185000,60.750000,62.381000
75%,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,9.515234,73.108995,86.929382,10652.310535,19440.143103,20319.402181,75.597000,79.443500,80.981500
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,52.000000,98.323610,100.000000,82365.492365,167313.266280,173688.189360,100.000000,100.000000,100.000000


**[TODO — bảng tóm tắt "trước / sau" (đây là bằng chứng chương trình hoá)]**

Bảng markdown liệt kê **số dòng, số cột, số thực thể, số chỉ số, số ô trống tại mỗi bước**:

| Bước | Rows | Cols | Thực thể | Chỉ số | % ô trống |
|---|---|---|---|---|---|
| Sau khi đọc `SDGData.csv` | ? | ? | ? | ? | ? |
| Sau khi drop cột thừa | ? | ? | ? | ? | ? |
| Sau khi lọc aggregate | ? | ? | ? | ? | ? |
| Sau khi lọc 5 chỉ số | ? | ? | ? | ? | ? |
| Sau khi melt (long) | ? | ? | ? | ? | ? |
| Bảng phân tích cuối | ? | ? | ? | ? | ? |

*Lấy con số thật từ output các cell code phía trên.*

## Exploratory Data Analysis

**[TODO — mô tả "bộ dữ liệu trông thế nào", CHỈ MÔ TẢ, chưa kết luận]**

Cấu trúc (217 nước × 5 chỉ số × 31 năm, 7 vùng, 4 nhóm thu nhập), độ phủ dữ liệu,
đơn vị từng chỉ số, cách nhóm (`Region` / `Income Group`).

In [9]:
# ── CELL 18: hồ sơ dữ liệu (profile) ──
print("Số quốc gia theo Region (217 nước thật):")
display(df_country.loc[df_country['is_country'], 'Region'].value_counts())

print("Số quốc gia theo Income Group (nhớ nhìn dòng NaN = Venezuela):")
display(df_country.loc[df_country['is_country'], 'Income Group'].value_counts(dropna=False))

print("Thiếu dữ liệu theo năm (subset 5 chỉ số, %):")
pct_missing_year = df_long.groupby('Year')['Value'].apply(
    lambda s: round(100 * s.isna().mean(), 1)
)
display(pct_missing_year.rename('% thiếu'))

print("\nThống kê mô tả bảng phân tích:")
display(tbl.describe())

Số quốc gia theo Region (217 nước thật):


Region
Europe & Central Asia         58
Sub-Saharan Africa            48
Latin America & Caribbean     42
East Asia & Pacific           37
Middle East & North Africa    21
South Asia                     8
North America                  3
Name: count, dtype: int64

Số quốc gia theo Income Group (nhớ nhìn dòng NaN = Venezuela):


Income Group
High income            80
Lower middle income    54
Upper middle income    54
Low income             28
NaN                     1
Name: count, dtype: int64

Thiếu dữ liệu theo năm (subset 5 chỉ số, %):


Year
1990    35.6
1991    34.6
1992    33.3
1993    32.3
1994    31.5
1995    30.0
1996    32.5
1997    31.1
1998    29.8
1999    28.5
2000     6.8
2001     6.5
2002     5.6
2003     6.2
2004     5.9
2005     5.8
2006     5.7
2007     5.2
2008     5.3
2009     5.2
2010     5.1
2011     4.6
2012     4.9
2013     4.9
2014     4.8
2015     5.0
2016     4.8
2017     4.7
2018     8.7
2019     9.3
2020    10.3
Name: % thiếu, dtype: float64


Thống kê mô tả bảng phân tích:


,EG.CFT.ACCS.ZS_2000,EG.CFT.ACCS.ZS_2015,EG.CFT.ACCS.ZS_2020,EG.ELC.ACCS.ZS_2000,EG.ELC.ACCS.ZS_2015,EG.ELC.ACCS.ZS_2020,IT.NET.USER.ZS_2000,IT.NET.USER.ZS_2015,IT.NET.USER.ZS_2020,NY.GDP.PCAP.CD_2000,NY.GDP.PCAP.CD_2015,NY.GDP.PCAP.CD_2020,SP.URB.TOTL.IN.ZS_2000,SP.URB.TOTL.IN.ZS_2015,SP.URB.TOTL.IN.ZS_2020
count,190.000000,190.000000,189.000000,211.000000,216.000000,215.000000,196.000000,200.000000,151.000000,199.000000,210.000000,203.000000,215.000000,215.000000,215.000000
mean,57.671368,65.581158,67.399735,75.672378,83.452441,86.953360,8.732304,48.034010,64.623466,8312.053771,16687.650764,16303.997091,55.491195,59.923451,61.415721
std,40.492401,38.468605,37.703206,33.498639,26.875029,23.605581,13.321321,28.997884,26.601769,13055.941231,24896.974671,23593.997636,24.612698,24.103121,23.858148
min,0.100000,0.000000,0.000000,1.613591,4.601340,7.241338,0.000000,1.083733,6.500000,124.460791,305.511144,233.837510,8.246000,12.078000,13.345000
25%,13.850000,26.137500,32.200000,54.012449,73.811436,86.136776,0.380223,20.916073,39.400000,651.272878,2213.957780,2262.250712,34.821000,40.914500,42.791500
50%,66.155000,85.450000,86.800000,97.021797,99.634331,100.000000,2.322545,49.927812,71.968194,2058.264401,6244.470178,6117.493571,55.185000,60.750000,62.381000
75%,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,9.515234,73.108995,86.929382,10652.310535,19440.143103,20319.402181,75.597000,79.443500,80.981500
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,52.000000,98.323610,100.000000,82365.492365,167313.266280,173688.189360,100.000000,100.000000,100.000000


### Quan sát 1

**[TODO — viết quan sát kèm CON SỐ CỤ THỂ trong văn bản]**

Gợi ý chủ đề: ***"Dữ liệu thiếu không ngẫu nhiên"*** — độ phủ tăng dần theo năm,
rơi mạnh ở 2020, và `EG.CFT.ACCS.ZS` hoàn toàn trống trước năm 2000.

*Bắt buộc:* nêu số (% ô trống, số nước có dữ liệu…) ngay trong đoạn văn + hình ở cell sau.

In [10]:
# ── CELL 20: hình cho Quan sát 1 ──
# [TODO] Vẽ biểu đồ số giá trị CÓ DỮ LIỆU theo năm (2 đường: toàn bộ SDGData vs subset 5 chỉ số)
# - trục x: Year, trục y: số giá trị / %
# - ghi chú "2020 chịu ảnh hưởng COVID" dưới hình (nguồn: SDGSeries-Time.csv)

### Quan sát 2

**[TODO — viết quan sát kèm CON SỐ CỤ THỂ]**

Gợi ý chủ đề: ***"Bất bình đẳng cơ sở hạ tầng đi theo nhóm thu nhập"*** —
xếp theo `Income Group` thấy thứ tự gần như cố định:
High > Upper middle > Lower middle > Low.

In [11]:
# ── CELL 22: hình cho Quan sát 2 ──
# [TODO] Boxplot HOẶC bar trung bình theo Income Group cho 2 chỉ số (điện + internet)
# - nhớ in SỐ NƯỚC trong từng nhóm (tránh trung bình trên 2-3 nước)

### Quan sát 3

**[TODO — viết quan sát kèm CON SỐ CỤ THỂ]**

Gợi ý chủ đề (kỹ thuật, rất được điểm): ***"Trung bình cộng ≠ giá trị World"*** —
`Aggregation method` của các chỉ số này là **Weighted average** (trọng số dân số),
nên simple mean của ~200 nước sẽ KHÁC con số mã `WLD`. Đây là cảnh báo sai lầm phân tích.

In [12]:
# ── CELL 24: đối chiếu simple mean vs giá trị World ──
# [TODO] In bảng: trung bình đơn giản của 217 nước vs giá trị mã 'WLD'
# cho 1-2 năm × 1-2 chỉ số. (Lấy WLD từ df_data[df_data['Country Code'] == 'WLD'])
# Kèm 1-2 câu diễn giải vì sao hai con số khác nhau (weighted average).

## Descriptive Analytics

**Quy tắc cứng cho mỗi câu hỏi: ① Câu hỏi → ② Vì sao câu hỏi này thú vị → ③ Trả lời bằng con số + hình.**
*(Phần ② hay bị bỏ qua nhất và bị trừ điểm.)*

### Câu hỏi 1

**① Câu hỏi [TODO]:** *Khoảng cách tiếp cận điện & internet giữa nhóm thu nhập cao nhất
và thấp nhất là bao nhiêu, và nó khác nhau thế nào giữa các Region?*

**② Vì sao thú vị [TODO]:** *viết 1–2 câu* — gợi ý: đây là chỉ tiêu đo bất bình đẳng
cơ sở hạ tầng (SDG 7 & 9); trả lời được "phân phối cơ hội tiếp cận" chứ không chỉ con số toàn cầu.

In [13]:
# ── Q1 (Cell 27): [TODO] ──
# [TODO] Bảng trung bình theo Income Group và theo Region tại NĂM CUỐI đã chốt (Cell 15)
# + biểu đồ cột có group.
# BẮT BUỘC: in số nước trong từng nhóm (n_groups) trước khi trình bày trung bình.

**③ Trả lời [TODO]:** nêu **con số cụ thể**
(nhóm cao nhất ≈ …%, nhóm thấp nhất ≈ …%, chênh lệch ≈ … điểm %) + 1 câu diễn giải.
*Không viết chung chung kiểu "khoảng cách khá lớn".

### Câu hỏi 2

**① Câu hỏi [TODO]:** *Từ 2000 đến 2020 (internet: 2015), khoảng cách đó THU HẸP hay
MỞ RỘNG? Tốc độ thu hẹp có giống nhau giữa các chỉ số?*

**② Vì sao thú vị [TODO]:** *viết 1–2 câu* — gợi ý: trả lời trực tiếp cho mục tiêu
"không để ai bị bỏ lại"; số điểm % thu hẹp là kết quả policy-relevant, dễ kể chuyện.

In [14]:
# ── Q2 (Cell 30): [TODO] ──
# [TODO] Tính trung bình theo Income Group tại 2 MỐC NĂM đã chốt →
# bảng GAP (High − Low) năm 1 và năm 2 → biểu đồ ĐƯỜNG theo Year cho từng nhóm.

**③ Trả lời [TODO]:** nêu gap năm 1, gap năm 2, **độ thay đổi theo điểm %**,
và chỉ ra nhóm nào thu hẹp nhanh nhất — bằng con số.

### Câu hỏi 3

**① Câu hỏi [TODO]:** *Những quốc gia nào cải thiện nhanh nhất và những quốc gia nào
thụt lùi về tiếp cận điện trong 2000–2020? Điều gì giải thích cho nhóm thụt lùi?*

**② Vì sao thú vị [TODO]:** *viết 1–2 câu* — gợi ý: chỉ số tổng quát (World/region)
che mất **outlier quốc gia**; tìm người bị bỏ lại là bước tiếp theo tự nhiên sau Q1–Q2.

In [15]:
# ── Q3 (Cell 33): [TODO] ──
# [TODO] delta = giá trị năm cuối − giá trị 2000 cho từng nước (chỉ số điện) →
# sort → top 10 cải thiện & bottom 5 thụt lùi → biểu đồ thanh ngang.
# Gợi ý thêm: so delta với nhóm thu nhập / GDP để có 1 nhận xét tương quan đơn giản.

**③ Trả lời [TODO]:** đích danh vài nước + **con số delta** cụ thể;
giải thích nhóm thụt lùi bằng bối cảnh (xung đột, thiên tai…) —
**nếu trích nguồn bên ngoài phải ghi nguồn** (rule C4).

## Suggestions

**[TODO — chọn 2–3 đề xuất, mỗi đề xuất phải có LUẬN CỨ — không bắt buộc cài đặt]**

Gợi ý (dùng chính hạn chế đã phát hiện làm luận cứ — thể hiện bạn đọc thấu dữ liệu):

1. **Nối dữ liệu ngoài:** `SDG Index (Our World in Data)` / `WDI` theo `CountryCode`
   → bổ sung biến dân số, đầu tư điện để giải thích gap.
2. **Mô hình hoá:** regression `log(GDP per capita) → access` để kiểm định thu nhập giải thích
   bao nhiêu %; hoặc dự báo giá trị thiếu có cơ sở thay vì bỏ trống.
3. **Dashboard:** chuyển bảng thành Power BI/Tableau cho bên ra quyết định.

In [16]:
# ── CELL 36 (tùy chọn nhưng nên có): minh hoạ tính khả thi ──
# [TODO] 1 thí nghiệm nhỏ, ví dụ:
# - ma trận tương quan (correlation) giữa 5 chỉ số, hoặc
# - scatter log(GDP per capita) vs access kèm đường xu hướng
# → nối với câu "mô hình hoá ở bước sau sẽ khả thi vì ...\"

## Bibliography

**[TODO — mọi nguồn đã dùng, kể cả nhỏ nhất]**

- **World Bank — Sustainable Development Goals Indicators** (nơi tải, ngày tải).
- **Nguồn metadata trong `SDGSeries.csv`**: ITU World Telecommunication Indicators,
  WHO Global Health Observatory, UN Population Division, World Bank Global Electrification Database…
- Tài liệu chính thức **pandas / matplotlib** (docs).
- Mọi blog, Stack Overflow, tài liệu khác, công cụ AI… bạn đã đọc.

**Quy tắc:** ① ghi nguồn **ngay chỗ dùng** (chú thích dưới hình/mục) + ② liệt kê ở đây;
③ nếu **dùng lại code của nguồn ngoài** → thêm 1 câu mô tả *"đã chỉnh gì cho khớp dữ liệu của mình"* (rule C5).